# Fluxa Voice Transaction Training Pipeline

Notebook ini dibuat untuk training dan evaluasi **transaction parser/classifier** Fluxa.

Batasan penting:

- Dataset 5000 JSON/JSONL dipakai untuk classifier transaksi, bukan untuk fine-tuning Whisper.
- Whisper small tetap dipakai sebagai pretrained speech-to-text di backend.
- Amount/nominal memakai rule-based parser agar stabil dan mudah dijelaskan di skripsi.
- Output model wajib menjadi draft transaksi, bukan langsung disimpan ke database.


## 0. Runtime Recommendation

- CPU Xxsmall cukup untuk validation, amount parser, dan baseline TF-IDF.
- GPU Xsmall disarankan untuk fine-tuning IndoBERT/XLM-R.
- Jalankan baseline dulu sebelum memakai GPU.


In [1]:
# Optional install cell. Uncomment ketika environment belum siap.
# %pip install -r ../requirements.txt


## 1. Imports & Path Setup


In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.dataset_loader import load_dataset, validate_dataset, save_json
from src.text_normalizer import normalize_text
from src.amount_parser import parse_amount
from src.metrics_utils import classification_metrics, evaluate_end_to_end

PROJECT_ROOT


PosixPath('/home/jovyan/projects/Fluxa/fluxa_voice_ai_training_pipeline')

## 2. Configuration

Ubah `DATASET_PATH` setelah kamu memindahkan dataset final ke folder `data/`.


In [3]:
EXPERIMENT_NAME = "fluxa_voice_intent_training_v1"
RANDOM_STATE = 42

DATASET_PATH = PROJECT_ROOT / "data" / "fluxa_voice_intent_dataset_sunda_5000.jsonl"
SAMPLE_DATASET_PATH = PROJECT_ROOT / "data" / "sample_dataset.jsonl"

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for directory in [MODEL_DIR, REPORT_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text"
TYPE_COL = "type"
CATEGORY_COL = "category"
WALLET_COL = "wallet"
AMOUNT_COL = "amount"
DESCRIPTION_COL = "description"

print("Experiment:", EXPERIMENT_NAME)
print("Dataset path:", DATASET_PATH)


Experiment: fluxa_voice_intent_training_v1
Dataset path: /home/jovyan/projects/Fluxa/fluxa_voice_ai_training_pipeline/data/fluxa_voice_intent_dataset_sunda_5000.jsonl


## 3. Load Dataset

Jika dataset final belum dipindahkan, notebook otomatis memakai `sample_dataset.jsonl` agar semua cell bisa dites.


In [4]:
active_dataset_path = DATASET_PATH if DATASET_PATH.exists() else SAMPLE_DATASET_PATH
print("Using dataset:", active_dataset_path)

df_raw = load_dataset(active_dataset_path)
print(df_raw.shape)
df_raw.head()


Using dataset: /home/jovyan/projects/Fluxa/fluxa_voice_ai_training_pipeline/data/fluxa_voice_intent_dataset_sunda_5000.jsonl
(5000, 9)


,language_style,raw_text,normalized_text,expected,difficulty,case_type,id,group,split
0,su,poé ieu meuli kuota internét sakitar tujuh pul...,poé ieu meuli kuota internét sakitar 75000,"{'amount': 75000, 'type': 'expense', 'category...",easy,expense_basic,VX-SU-000001,expense_basic,train
1,su,abdi mayar saratus rébu keur dokter,abdi mayar 100000 keur dokter,"{'amount': 100000, 'type': 'expense', 'categor...",easy,expense_basic,VX-SU-000002,expense_basic,train
2,su,meuli dua belas rebu keur roti,meuli 12000 keur roti,"{'amount': 12000, 'type': 'expense', 'category...",easy,expense_basic,VX-SU-000003,expense_basic,train
3,su,urang meuli bensin sakitar 30rb,urang meuli bensin sakitar 30000,"{'amount': 30000, 'type': 'expense', 'category...",easy,expense_basic,VX-SU-000004,expense_basic,train
4,su,minggu ieu bioskop 30rb,minggu ieu bioskop 30000,"{'amount': 30000, 'type': 'expense', 'category...",easy,expense_basic,VX-SU-000005,expense_basic,train


## 4. Dataset Validation

Validasi memastikan data siap training dan menyimpan summary ke `reports/dataset_summary.json`.


In [5]:
df_valid, df_invalid, dataset_summary = validate_dataset(df_raw)

save_json(dataset_summary, REPORT_DIR / "dataset_summary.json")
if not df_invalid.empty:
    df_invalid.to_csv(REPORT_DIR / "invalid_rows.csv", index=False)

print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))
print("Invalid rows:", len(df_invalid))
df_valid.head()


ValueError: Missing required columns: ['text', 'type']

## 5. Cleaning & Normalization

Normalizer dibuat konservatif agar variasi Sunda/Indonesia tetap terbaca oleh model.


In [ ]:
df = df_valid.copy()
df["normalized_text"] = df[TEXT_COL].map(normalize_text)

df[[TEXT_COL, "normalized_text", TYPE_COL, CATEGORY_COL, AMOUNT_COL]].head(10)


## 6. Exploratory Data Analysis


In [ ]:
print("Rows:", len(df))
print("Duplicate normalized texts:", df["normalized_text"].duplicated().sum())
print("Type distribution:")
print(df[TYPE_COL].value_counts())

if CATEGORY_COL in df.columns:
    print("
Top categories:")
    print(df[CATEGORY_COL].value_counts().head(20))

if WALLET_COL in df.columns:
    print("
Wallet null ratio:", float(df[WALLET_COL].isna().mean()))


In [ ]:
# Simple plots. No custom colors to keep this notebook portable.
df[TYPE_COL].value_counts().plot(kind="bar", title="Type Distribution")
plt.xlabel("type")
plt.ylabel("count")
plt.show()

if CATEGORY_COL in df.columns:
    df[CATEGORY_COL].value_counts().head(15).plot(kind="bar", title="Top 15 Category Distribution")
    plt.xlabel("category")
    plt.ylabel("count")
    plt.show()


## 7. Amount Parser Evaluation

Nominal uang diproses rule-based, bukan classifier. Ini lebih stabil untuk ekspresi seperti `lima rebu`, `35rb`, dan `dua juta`.


In [ ]:
df["pred_amount"] = df["normalized_text"].map(parse_amount)

if AMOUNT_COL in df.columns:
    amount_eval = df.dropna(subset=[AMOUNT_COL]).copy()
    amount_eval["amount_correct"] = amount_eval[AMOUNT_COL].astype("Int64") == amount_eval["pred_amount"].astype("Int64")
    amount_metrics = {
        "samples": int(len(amount_eval)),
        "exact_match_accuracy": float(amount_eval["amount_correct"].mean()) if len(amount_eval) else None,
    }
    save_json(amount_metrics, REPORT_DIR / "amount_parser_metrics.json")
    amount_eval.loc[~amount_eval["amount_correct"], [TEXT_COL, "normalized_text", AMOUNT_COL, "pred_amount"]].to_csv(
        REPORT_DIR / "amount_parser_errors.csv", index=False
    )
    print(json.dumps(amount_metrics, indent=2))
    display(amount_eval[[TEXT_COL, AMOUNT_COL, "pred_amount", "amount_correct"]].head(20))
else:
    print("No amount column found; skipping amount evaluation.")


## 8. Train / Validation / Test Split

Split default: 80% train, 10% validation, 10% test. Untuk dataset kecil/sample, stratify bisa gagal; notebook akan fallback otomatis.


In [ ]:
required_for_training = ["normalized_text", TYPE_COL, CATEGORY_COL]
missing = [col for col in required_for_training if col not in df.columns]
if missing:
    raise ValueError(f"Missing required training columns: {missing}")

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df[TYPE_COL] if df[TYPE_COL].value_counts().min() >= 2 else None,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df[TYPE_COL] if temp_df[TYPE_COL].value_counts().min() >= 2 else None,
)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)


## 9. Label Mapping Export


In [ ]:
type_labels = sorted(df[TYPE_COL].dropna().unique().tolist())
category_labels = sorted(df[CATEGORY_COL].dropna().unique().tolist())
wallet_labels = sorted(df[WALLET_COL].dropna().unique().tolist()) if WALLET_COL in df.columns else []

label_maps = {
    "type2id": {label: i for i, label in enumerate(type_labels)},
    "id2type": {i: label for i, label in enumerate(type_labels)},
    "category2id": {label: i for i, label in enumerate(category_labels)},
    "id2category": {i: label for i, label in enumerate(category_labels)},
    "wallet2id": {label: i for i, label in enumerate(wallet_labels)},
    "id2wallet": {i: label for i, label in enumerate(wallet_labels)},
}

save_json(label_maps, MODEL_DIR / "label_mappings.json")
print(json.dumps(label_maps, ensure_ascii=False, indent=2))


## 10. Baseline Classifier: TF-IDF + Logistic Regression

Baseline wajib ada sebelum fine-tuning transformer. Ini cepat, murah, dan menjadi pembanding akademik.


In [ ]:
def build_baseline_classifier(max_features: int = 12000) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=1)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])

type_model = build_baseline_classifier()
category_model = build_baseline_classifier()

type_model.fit(train_df["normalized_text"], train_df[TYPE_COL])
category_model.fit(train_df["normalized_text"], train_df[CATEGORY_COL])

joblib.dump(type_model, MODEL_DIR / "baseline_type_classifier.joblib")
joblib.dump(category_model, MODEL_DIR / "baseline_category_classifier.joblib")

print("Baseline models saved.")


## 11. Baseline Evaluation


In [ ]:
test_eval = test_df.copy()
test_eval["pred_type"] = type_model.predict(test_eval["normalized_text"])
test_eval["pred_category"] = category_model.predict(test_eval["normalized_text"])
test_eval["pred_amount"] = test_eval["normalized_text"].map(parse_amount)

type_metrics = classification_metrics(test_eval[TYPE_COL], test_eval["pred_type"])
category_metrics = classification_metrics(test_eval[CATEGORY_COL], test_eval["pred_category"])

save_json(type_metrics, REPORT_DIR / "baseline_type_metrics.json")
save_json(category_metrics, REPORT_DIR / "baseline_category_metrics.json")

test_eval.to_csv(REPORT_DIR / "baseline_test_predictions.csv", index=False)

print("Type metrics:")
print(json.dumps({k: v for k, v in type_metrics.items() if k != "classification_report"}, indent=2))
print("
Category metrics:")
print(json.dumps({k: v for k, v in category_metrics.items() if k != "classification_report"}, indent=2))


## 12. End-to-End Evaluation


In [ ]:
if AMOUNT_COL in test_eval.columns:
    e2e_metrics = evaluate_end_to_end(test_eval)
    save_json(e2e_metrics, REPORT_DIR / "end_to_end_metrics.json")
    print(json.dumps(e2e_metrics, indent=2))
else:
    print("No amount column found; E2E amount evaluation skipped.")


## 13. Transformer Fine-Tuning Template: IndoBERT/XLM-R

Jalankan section ini hanya saat memakai GPU. Untuk eksperimen awal, baseline sudah cukup.

Rekomendasi:

- IndoBERT: `indobenchmark/indobert-base-p1`
- XLM-R: `xlm-roberta-base`


In [ ]:
RUN_TRANSFORMER_TRAINING = False
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 96
NUM_EPOCHS = 5

print("Transformer training enabled:", RUN_TRANSFORMER_TRAINING)


In [ ]:
if RUN_TRANSFORMER_TRAINING:
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
    )

    target_col = TYPE_COL  # Change to CATEGORY_COL for category classifier.
    labels = sorted(df[target_col].dropna().unique().tolist())
    label2id = {label: idx for idx, label in enumerate(labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    hf_train = Dataset.from_pandas(train_df[["normalized_text", target_col]].rename(columns={target_col: "label"}))
    hf_val = Dataset.from_pandas(val_df[["normalized_text", target_col]].rename(columns={target_col: "label"}))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def preprocess(batch):
        tokenized = tokenizer(batch["normalized_text"], truncation=True, max_length=MAX_LENGTH)
        tokenized["labels"] = [label2id[x] for x in batch["label"]]
        return tokenized

    hf_train = hf_train.map(preprocess, batched=True)
    hf_val = hf_val.map(preprocess, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / "transformer_type_classifier"),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=25,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()
    trainer.save_model(str(MODEL_DIR / "transformer_type_classifier"))
    tokenizer.save_pretrained(str(MODEL_DIR / "transformer_type_classifier"))
    save_json({"label2id": label2id, "id2label": id2label}, MODEL_DIR / "transformer_type_classifier" / "label_mapping.json")
else:
    print("Skipped transformer training.")


## 14. Local Inference Demo

Cell ini menunjukkan output yang nanti dikembalikan backend FastAPI ke Flutter.


In [ ]:
def predict_transaction_json(text: str) -> dict:
    normalized = normalize_text(text)
    pred_type = type_model.predict([normalized])[0]
    pred_category = category_model.predict([normalized])[0]
    pred_amount = parse_amount(normalized)

    return {
        "transcript": {
            "raw": text,
            "normalized": normalized,
            "language_hint": "su-id",
            "confidence": 0.0,
        },
        "transaction": {
            "type": pred_type,
            "amount": pred_amount,
            "category": pred_category,
            "wallet": None,
            "description": normalized,
            "currency": "IDR",
        },
        "classification": {
            "type_confidence": 0.0,
            "category_confidence": 0.0,
            "wallet_confidence": 0.0,
            "overall_confidence": 0.0,
        },
        "warnings": [],
    }

examples = [
    "mayar parkir motor lima rebu",
    "beli kopi 35rb dari bca",
    "nampi gajih dua juta",
]

for example in examples:
    print(json.dumps(predict_transaction_json(example), ensure_ascii=False, indent=2))


## 15. Final Summary

Setelah notebook selesai dijalankan, cek:

```text
models/baseline_type_classifier.joblib
models/baseline_category_classifier.joblib
models/label_mappings.json
reports/dataset_summary.json
reports/baseline_type_metrics.json
reports/baseline_category_metrics.json
reports/amount_parser_metrics.json
reports/end_to_end_metrics.json
```

Jika baseline sudah stabil, baru lanjut fine-tuning IndoBERT/XLM-R di GPU Xsmall.
